In [1]:
from operator import itemgetter

from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.documents import Document
from flashrank import Ranker, RerankRequest

In [2]:
DATA_DIR = "./RAG_data"

txt_loader = DirectoryLoader(DATA_DIR, glob="./*.txt", loader_cls=TextLoader)
pdf_loader = DirectoryLoader(DATA_DIR, glob="./*.pdf", loader_cls=PyPDFLoader)
docs = txt_loader.load() + pdf_loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
splits = text_splitter.split_documents(docs)

print(f"Loaded {len(docs)} documents → {len(splits)} chunks")

Loaded 73 documents → 302 chunks


In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 5

def hybrid_retrieve(query: str) -> list:
    vec_docs = vector_retriever.invoke(query)
    bm25_docs = bm25_retriever.invoke(query)
    seen = {}
    for d in vec_docs + bm25_docs:
        if d.page_content not in seen:
            seen[d.page_content] = d
    return list(seen.values())

print("Hybrid retriever ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hybrid retriever ready


In [6]:
llm = ChatOllama(model="llama3.2", temperature=0)

# Rewrite follow-up questions to be standalone before retrieval
CONTEXTUALIZE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, rewrite the question "
     "to be fully self-contained. Do NOT answer it. Only reformulate if needed, "
     "otherwise return it as-is."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])
contextualize_chain = CONTEXTUALIZE_PROMPT | llm | StrOutputParser()

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise research assistant. Answer ONLY from the provided context.\n"
     "If the answer is not in the context, say 'I don't have enough information.'\n"
     "Cite the source filename for every claim. Be concise."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

# Query expansion
expansion_prompt = ChatPromptTemplate.from_template(
    "Generate 3 short search queries related to: {question}\nOutput only the queries, one per line."
)
expansion_chain = expansion_prompt | llm | StrOutputParser()

# Reranker
ranker = Ranker(model_name="ms-marco-MultiBERT-L-12")

def rerank(query, docs, top_k=4):
    if not docs:
        return []
    passages = [{"id": i, "text": d.page_content, "meta": d.metadata} for i, d in enumerate(docs)]
    results = ranker.rerank(RerankRequest(query=query, passages=passages))
    return [Document(page_content=r["text"], metadata=r["meta"]) for r in results[:top_k]]

def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata.get('source', 'Unknown')}]\n{d.page_content}" for d in docs)

def retrieve_and_rerank(inputs):
    question = inputs["question"]
    history = inputs["chat_history"]

    # Rewrite question with context so retriever gets a good query
    standalone_q = contextualize_chain.invoke(
        {"question": question, "chat_history": history}
    ) if history else question

    # Expand + retrieve
    expanded = expansion_chain.invoke({"question": standalone_q})
    queries = [standalone_q] + [q.strip() for q in expanded.strip().split("\n") if q.strip()]

    all_docs = []
    for q in queries:
        all_docs.extend(hybrid_retrieve(q))

    unique_docs = list({d.page_content: d for d in all_docs}.values())
    return format_docs(rerank(standalone_q, unique_docs))

# Final chain
rag_chain = (
    {
        "context": retrieve_and_rerank,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | ANSWER_PROMPT
    | llm
    | StrOutputParser()
)

print("Chain ready")

Chain ready


In [7]:
MAX_HISTORY_TURNS = 5
chat_history = []

def ask(query):
    global chat_history
    trimmed = chat_history[-(MAX_HISTORY_TURNS * 2):]
    result = rag_chain.invoke({"question": query, "chat_history": trimmed})
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=result))
    return result

def reset():
    global chat_history
    chat_history = []
    print("History cleared.")

In [8]:
reset()

questions = [
    "What are the energy transfer strategies?",
    "What is the novelty in using NARS scheduler?",
    "What did they mean by SINDy+RL?",
    "How did they reduce the number of communications?",
    "How did they reduce the number of communications in the energy-accuracy paper?",
]

for i, q in enumerate(questions, 1):
    print(f"\n{'='*60}")
    print(f"Q{i}: {q}")
    print(f"{'='*60}")
    print(ask(q))

History cleared.

Q1: What are the energy transfer strategies?


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


According to the provided context, there are two main energy transfer strategies:

1. Centralized Mechanism:
In this mechanism, the OBN maintains the energy status of all nodes.

2. Decentralized Mechanism:
In this mechanism, each node determines its own energy needs and makes autonomous decisions during energy transfers without waiting for specific invitations from the OBN.

Q2: What is the novelty in using NARS scheduler?


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


I don't have enough information about what you are referring to, but according to [Source: RAG_data/CIS5590-Project-2.pdf], the novelty of using NARS scheduler lies in its ability to outperform traditional learning rate schedulers such as OneCycle and Fixed on various datasets.

Q3: What did they mean by SINDy+RL?


...

According to [Source: RAG_data/AI-5-2.pdf], SINDy+RL refers to the integration of Sparse Identification of Nonlinear Dynamics (SINDy) with reinforcement learning (RL). In this approach, SINDy is used to learn compact surrogate models that drive RL-style control of nonlinear systems.

Q4: How did they reduce the number of communications?


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


I don't have enough information about how they reduced the number of communications. The provided context only mentions that a simple scheduled media-access layer (MAC) for CDMN is assumed, with every message type assigned a specific time-slot to be transmitted with minimum delay. However, it does not provide further details on how this approach reduces the number of communications.

Q5: How did they reduce the number of communications in the energy-accuracy paper?


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


According to [Source: RAG_data/Energy_Transfer_Strategies_in_Magnetic_Resonance_Based_Intrabody_Networks.pdf], they reduced the number of communications by using a simple scheduled media-access layer (MAC) for CDMN. Every message type is assigned a specific time-slot so that it can be transmitted with minimum delay, and all time intervals are assumed to be discretized and equal.


## RAG V2 — Technical Summary

**Architecture:** Local-first RAG pipeline using LangChain, Ollama (Llama 3.2), ChromaDB, and FlashRank.

**Pipeline:**
1. **Load** — PDFs and TXT files via DirectoryLoader
2. **Chunk** — RecursiveCharacterTextSplitter (1000 chars, 200 overlap)
3. **Contextualize** — Follow-up questions are rewritten as standalone queries using chat history before retrieval
4. **Expand** — Each query is expanded into 3 related search queries for broader recall
5. **Retrieve** — Hybrid search: BM25 (keyword) + ChromaDB (semantic), deduplicated
6. **Rerank** — FlashRank cross-encoder selects top 4 chunks
7. **Generate** — Llama 3.2 answers strictly from retrieved context with source citations

**Key design decisions:**
- Chat history trimmed to last 5 turns to avoid context overflow
- Hybrid retrieval ensures both exact keyword matches and semantic similarity are captured
- Query contextualization enables multi-turn conversations with accurate retrieval

In [15]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated

llm = ChatOllama(model="llama3.2", temperature=0)

# Prompts (same as before)
CONTEXTUALIZE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, rewrite the question "
     "to be fully self-contained. Do NOT answer it. Only reformulate if needed, "
     "otherwise return it as-is."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])
contextualize_chain = CONTEXTUALIZE_PROMPT | llm | StrOutputParser()

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise research assistant. Answer ONLY from the provided context.\n"
     "If the answer is not in the context, say 'I don't have enough information.'\n"
     "Cite the source filename for every claim. Be concise."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

expansion_prompt = ChatPromptTemplate.from_template(
    "Generate 3 short search queries related to: {question}\nOutput only the queries, one per line."
)
expansion_chain = expansion_prompt | llm | StrOutputParser()

ranker = Ranker(model_name="ms-marco-MultiBERT-L-12")

class RAGState(TypedDict):
    question: str
    chat_history: list
    standalone_question: str
    queries: list[str]
    documents: list
    context: str
    answer: str
    retry_count: int

'''# State schema — this is what flows between nodes
class RAGState(TypedDict):
    question: str
    chat_history: list
    standalone_question: str
    queries: list[str]
    documents: list
    context: str
    answer: str'''

'# State schema — this is what flows between nodes\nclass RAGState(TypedDict):\n    question: str\n    chat_history: list\n    standalone_question: str\n    queries: list[str]\n    documents: list\n    context: str\n    answer: str'

In [22]:
def contextualize(state: RAGState) -> dict:
    """Rewrite question to be standalone if there's history."""
    if state["chat_history"]:
        standalone = contextualize_chain.invoke({
            "question": state["question"],
            "chat_history": state["chat_history"],
        })
    else:
        standalone = state["question"]
    return {"standalone_question": standalone}


def expand_query(state: RAGState) -> dict:
    """Generate related queries for broader recall."""
    expanded = expansion_chain.invoke({"question": state["standalone_question"]})
    queries = [state["standalone_question"]] + [
        q.strip() for q in expanded.strip().split("\n") if q.strip()
    ]
    return {"queries": queries}


'''def retrieve(state: RAGState) -> dict:
    """Hybrid retrieve + deduplicate."""
    all_docs = []
    for q in state["queries"]:
        all_docs.extend(hybrid_retrieve(q))
    unique = list({d.page_content: d for d in all_docs}.values())
    return {"documents": unique}'''


def retrieve(state: RAGState) -> dict:
    """Hybrid retrieve + deduplicate. Tracks retry count."""
    all_docs = []
    for q in state["queries"]:
        all_docs.extend(hybrid_retrieve(q))
    unique = list({d.page_content: d for d in all_docs}.values())
    retries = state.get("retry_count", 0)
    return {"documents": unique, "retry_count": retries}

def rerank_docs(state: RAGState) -> dict:
    """Rerank and format as context string."""
    docs = state["documents"]
    if not docs:
        return {"context": "No relevant documents found."}
    passages = [{"id": i, "text": d.page_content, "meta": d.metadata} for i, d in enumerate(docs)]
    results = ranker.rerank(RerankRequest(query=state["standalone_question"], passages=passages))
    top_docs = [Document(page_content=r["text"], metadata=r["meta"]) for r in results[:4]]
    context = "\n\n".join(
        f"[Source: {d.metadata.get('source', 'Unknown')}]\n{d.page_content}" for d in top_docs
    )
    return {"context": context}

def check_quality(state: RAGState) -> dict:
    retries = state.get("retry_count", 0)
    if retries >= 2:
        return {}
    
    # Ask LLM: does the context answer the question?
    check = llm.invoke(
        f"Does this context contain enough information to answer the question?\n"
        f"Question: {state['standalone_question']}\n"
        f"Context: {state['context'][:500]}\n"
        f"Reply ONLY 'yes' or 'no'."
    ).content.strip().lower()
    
    if "no" in check:
        broader = expansion_chain.invoke({
            "question": f"Broaden this search, find related concepts: {state['standalone_question']}"
        })
        new_queries = [q.strip() for q in broader.strip().split("\n") if q.strip()]
        print(f"  ↻ Retry {retries + 1}: context insufficient, broadening...")
        return {"queries": new_queries, "retry_count": retries + 1}
    return {}

def generate(state: RAGState) -> dict:
    """Generate final answer from context."""
    chain = ANSWER_PROMPT | llm | StrOutputParser()
    answer = chain.invoke({
        "context": state["context"],
        "question": state["question"],
        "chat_history": state["chat_history"],
    })
    return {"answer": answer}

In [23]:
'''graph = StateGraph(RAGState)

# Add nodes
graph.add_node("contextualize", contextualize)
graph.add_node("expand_query", expand_query)
graph.add_node("retrieve", retrieve)
graph.add_node("rerank", rerank_docs)
graph.add_node("generate", generate)

# Wire edges
graph.set_entry_point("contextualize")
graph.add_edge("contextualize", "expand_query")
graph.add_edge("expand_query", "retrieve")
graph.add_edge("retrieve", "rerank")
graph.add_edge("rerank", "generate")
graph.add_edge("generate", END)

# Compile
rag_graph = graph.compile()
print("Graph compiled")'''

graph = StateGraph(RAGState)

graph.add_node("contextualize", contextualize)
graph.add_node("expand_query", expand_query)
graph.add_node("retrieve", retrieve)
graph.add_node("rerank", rerank_docs)
graph.add_node("check_quality", check_quality)
graph.add_node("generate", generate)

graph.set_entry_point("contextualize")
graph.add_edge("contextualize", "expand_query")
graph.add_edge("expand_query", "retrieve")
graph.add_edge("retrieve", "rerank")
graph.add_edge("rerank", "check_quality")

# Conditional: retry or proceed
graph.add_conditional_edges(
    "check_quality",
    lambda state: "retry" if state.get("retry_count", 0) > 0 and len(state["context"]) < 200 else "done",
    {"retry": "retrieve", "done": "generate"},
)
graph.add_edge("generate", END)

rag_graph = graph.compile()
print("Graph compiled (with retry loop)")

Graph compiled (with retry loop)


In [24]:
MAX_HISTORY_TURNS = 5
chat_history = []

'''def ask(query):
    global chat_history
    trimmed = chat_history[-(MAX_HISTORY_TURNS * 2):]

    result = rag_graph.invoke({
        "question": query,
        "chat_history": trimmed,
        "standalone_question": "",
        "queries": [],
        "documents": [],
        "context": "",
        "answer": "",
    })

    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=result["answer"]))
    return result["answer"]'''

def ask(query):
    global chat_history
    trimmed = chat_history[-(MAX_HISTORY_TURNS * 2):]

    result = rag_graph.invoke({
        "question": query,
        "chat_history": trimmed,
        "standalone_question": "",
        "queries": [],
        "documents": [],
        "context": "",
        "answer": "",
        "retry_count": 0,
    })

    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=result["answer"]))
    return result["answer"]

def reset():
    global chat_history
    chat_history = []
    print("History cleared.")

In [25]:
reset()

questions = [
    "What are the energy transfer strategies?",
    "What is the novelty in using NARS scheduler?",
    "What did they mean by SINDy+RL?",
    "How did they reduce the number of communications?",
    "How did they reduce the number of communications in the energy-accuracy paper?",
]

for i, q in enumerate(questions, 1):
    print(f"\n{'='*60}")
    print(f"Q{i}: {q}")
    print(f"{'='*60}")
    print(ask(q))

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


History cleared.

Q1: What are the energy transfer strategies?


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


According to the provided context, there are two main energy transfer strategies:

1. Centralized Mechanism:
In this mechanism, the OBN maintains the energy status of all nodes.

2. Decentralized Mechanism:
In this mechanism, each node determines its own energy needs and makes autonomous decisions during energy transfers without waiting for specific invitations from the OBN.

Q2: What is the novelty in using NARS scheduler?


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


...

I don't have enough information about how they reduced the number of communications. The provided context only mentions that a simple scheduled media-access layer (MAC) for CDMN is assumed, with every message type assigned a specific time-slot to be transmitted with minimum delay. However, it does not provide further details on how this approach reduces the number of communications.

Q5: How did they reduce the number of communications in the energy-accuracy paper?


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


  ↻ Retry 1: context insufficient, broadening...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


According to [Source: RAG_data/Energy_Transfer_Strategies_in_Magnetic_Resonance_Based_Intrabody_Networks.pdf], they reduced the number of communications by using a simple scheduled media-access layer (MAC) for CDMN. Every message type is assigned a specific time-slot so that it can be transmitted with minimum delay, and all time intervals are assumed to be discretized and equal.


In [26]:
print(ask("What is the quantum banana protocol?"))

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


  ↻ Retry 1: context insufficient, broadening...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


I don't have enough information about what you're referring to, as there is no mention of a "quantum banana protocol" in the provided context.
